In [2]:
from sentence_transformers import SentenceTransformer

In [4]:
from rank_bm25 import BM25Okapi
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

def generate_embedding(text):
    if isinstance(text, list):
        return [model.encode(t).tolist() for t in text]
    return model.encode(text).tolist()

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Vector Index
Stores embeddings and searches by semantic similarity.

In [5]:
class VectorIndex:
    def __init__(self):
        self.vectors = []
        self.documents = []

    def add_document(self, doc):
        self.vectors.append(generate_embedding(doc["content"]))
        self.documents.append(doc)

    def search(self, query, top_k=5):
        query_emb = generate_embedding(query)
        scores = [cosine_similarity(query_emb, v) for v in self.vectors]
        ranked = sorted(zip(scores, self.documents), reverse=True)
        return [(doc, score) for score, doc in ranked[:top_k]]

## BM25 Index
Searches by exact term matching, weighted by term rarity.

In [11]:
class BM25Index:
    def __init__(self):
        self.documents = []
        self.bm25 = None

    def add_document(self, doc):
        self.documents.append(doc)
        tokenized = [d["content"].lower().split() for d in self.documents]
        self.bm25 = BM25Okapi(tokenized)

    def search(self, query, top_k=5):
        tokens = query.lower().split()
        scores = self.bm25.get_scores(tokens)
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return [(self.documents[i], score) for i, score in ranked[:top_k]]

## Retriever
Combines both indexes using Reciprocal Rank Fusion.
Each chunk gets a rank from each index. RRF converts ranks into a unified score and sorts the final list.

In [12]:
class Retriever:
    def __init__(self, *indexes):
        if not indexes:
            raise ValueError("Provide at least one index")
        self.indexes = list(indexes)

    def add_document(self, doc):
        for index in self.indexes:
            index.add_document(doc)

    def search(self, query, top_k=3, k_rrf=1):
        rrf_scores = {}

        for index in self.indexes:
            results = index.search(query, top_k=10)
            for rank, (doc, _) in enumerate(results):
                key = doc["content"][:50]
                if key not in rrf_scores:
                    rrf_scores[key] = {"score": 0, "doc": doc}
                rrf_scores[key]["score"] += 1 / (k_rrf + rank + 1)

        ranked = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)
        return [(item["doc"], item["score"]) for item in ranked[:top_k]]

## Test It
Build the retriever, add some chunks, and search.

In [13]:
chunks = [
    "Medical Research: This year saw significant strides in understanding XDR-47, a bug we have not seen before.",
    "Software Engineering: This division studied infection vectors in distributed systems. Incident INC-2023-Q4-011 was resolved.",
    "Cybersecurity: INC-2023-Q4-011 was a critical breach affecting three servers in Q4.",
    "Financial Analysis: Revenue grew 12% year over year driven by enterprise contracts.",
]

retriever = Retriever(VectorIndex(), BM25Index())
for chunk in chunks:
    retriever.add_document({"content": chunk})

In [14]:
query = "What happened with INC-2023-Q4-011?"
results = retriever.search(query, top_k=3)

for doc, score in results:
    print(f"RRF Score: {score:.3f}")
    print(doc["content"][:200])
    print()

RRF Score: 0.750
Cybersecurity: INC-2023-Q4-011 was a critical breach affecting three servers in Q4.

RRF Score: 0.750
Medical Research: This year saw significant strides in understanding XDR-47, a bug we have not seen before.

RRF Score: 0.667
Software Engineering: This division studied infection vectors in distributed systems. Incident INC-2023-Q4-011 was resolved.

